*Functions used:* `os.makedirs()`, `kagglehub.dataset_download()`, `pd.read_csv()`, `os.listdir()`, `os.path.join()`, `pivot_table()`, `merge()`, `dropna()`, `reset_index()`, `describe()`, `to_csv()`

# Data Import & Preprocessing

Loads FBI UCR crime data and ACS 2023 socioeconomic files, merges them on FIPS county codes, and saves the analysis-ready dataset to `processed_data.csv` 

## 1. Packages & Data

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os

os.makedirs('output', exist_ok=True)

# Crime data (FBI UCR via Kaggle)

path     = kagglehub.dataset_download("mikejohnsonjr/united-states-crime-rates-by-county")
crime_df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))
print(f"Crime data:  {crime_df.shape[0]:,} rows × {crime_df.shape[1]} columns")

# ACS socioeconomic datasets 

acs_unem = pd.read_csv('Unemployment2023.csv')
acs_pov  = pd.read_csv('Poverty2023.csv')
acs_educ = pd.read_csv('Education2023.csv', encoding='latin1')
acs_educ = acs_educ.rename(columns={"FIPS Code": "FIPS_Code"})
print(f"Unemployment: {acs_unem.shape[0]:,} rows | Poverty: {acs_pov.shape[0]:,} rows | Education: {acs_educ.shape[0]:,} rows")

Crime data:  3,136 rows × 24 columns
Unemployment: 329,726 rows | Poverty: 79,961 rows | Education: 169,245 rows


## 2. Reshape & Merge ACS Data

In [ ]:
# to pivot each ACS dataset from long to wide 
unem_wide = acs_unem.pivot_table(index="FIPS_Code", columns="Attribute",
                                  values="Value", aggfunc="first").reset_index()

pov_wide  = acs_pov.pivot_table( index="FIPS_Code", columns="Attribute",
                                  values="Value", aggfunc="first").reset_index()

educ_wide = acs_educ.pivot_table(index="FIPS_Code", columns="Attribute",
                                  values="Value", aggfunc="first").reset_index()

acs = (unem_wide.merge(pov_wide, on="FIPS_Code", how="inner")
                .merge(educ_wide, on="FIPS_Code", how="inner"))
print(f"ACS merged:  {acs.shape[0]:,} counties × {acs.shape[1]} columns")

ACS merged:  3,195 counties × 185 columns


## 3. Merge Crime + ACS on FIPS Code

In [4]:
crime_df["FIPS_Code"] = (crime_df["FIPS_ST"].astype(str).str.zfill(2) +
                          crime_df["FIPS_CTY"].astype(str).str.zfill(3))
acs["FIPS_Code"]      = acs["FIPS_Code"].astype(str).str.zfill(5)

final_df = crime_df.merge(acs, on="FIPS_Code", how="inner")
print(f"Merged dataset: {final_df.shape[0]:,} rows × {final_df.shape[1]} columns")

Merged dataset: 3,123 rows × 209 columns


## 4. Select Analysis Variables & Drop Missing

In [ ]:
OUTCOME  = 'crime_rate_per_100000'
UNEM     = 'Unemployment_rate_2023'
POV      = 'PCTPOVALL_2023'
CONTROLS = ['MEDHHINC_2023',
    "Percent of adults who are not high school graduates, 2019-23",
    "Percent of adults with a bachelor's degree or higher, 2019-23",]

FEATURES = [UNEM, POV] + [c for c in CONTROLS if c in final_df.columns]

id_cols     = [c for c in ['county_name', 'FIPS_Code'] if c in final_df.columns]
analysis_df = final_df[id_cols + [OUTCOME] + FEATURES].dropna().copy().reset_index(drop=True)

print(f"Analysis dataset: {len(analysis_df):,} counties × {analysis_df.shape[1]} columns")
print(f"Rows dropped (missing values): {len(final_df) - len(analysis_df):,}")
print()
print(analysis_df[[OUTCOME, UNEM, POV]].describe().round(2))

Analysis dataset: 3,123 counties × 8 columns
Rows dropped (missing values): 0

       crime_rate_per_100000  Unemployment_rate_2023  PCTPOVALL_2023
count                3123.00                 3123.00         3123.00
mean                  235.48                    3.58           14.50
std                   200.66                    1.20            5.53
min                     0.00                    0.30            3.30
25%                    95.08                    2.80           10.60
50%                   186.25                    3.40           13.60
75%                   321.82                    4.10           17.40
max                  1792.00                   17.30           49.60


## 5. Save Processed Data

In [6]:
analysis_df.to_csv('processed_data.csv', index=False)
print("Saved: processed_data.csv")
print(f"Columns: {list(analysis_df.columns)}")

Saved: processed_data.csv
Columns: ['county_name', 'FIPS_Code', 'crime_rate_per_100000', 'Unemployment_rate_2023', 'PCTPOVALL_2023', 'MEDHHINC_2023', 'Percent of adults who are not high school graduates, 2019-23', "Percent of adults with a bachelor's degree or higher, 2019-23"]
